In [1]:
from Code.NetworkExpansion import *
from Code.externalize_results import *
import networkx as nx
import matplotlib.pyplot as plt
import scipy.stats as stats

# Expansions from primordial seed sets
write_outputs = True
max_generation = 80

# define chemical scope
reaction_scope = get_sides(import_reaction_file(rxn_db_dir+'kegg_data_R_processed.csv.zip'))
compound_scope = set.union(*reaction_scope.stack())
seed_scope = seed_candidates.loc[seed_candidates.index.isin(compound_scope)]
seed_groups = ['JM', 'G17', 'G24', 'JM|G24|G17|TW']
seed_sets = {i: seed_scope.query('Source.str.contains("'+i+'")').index for i in seed_groups}

# run network expansions
results = dict()
for name, seeds in seed_sets.items():
    results[name] = expand_from_seed_set(seeds=seeds, reactions=reaction_scope, max_gen=max_generation)
    results[name]['bipartite_edgelist'] = make_bipartite_edgelist(results[name]['edge_pool'])
    results[name]['arrival_step'] = results[name]['participants'].stack().explode().drop_duplicates().dropna().reset_index().set_index(0)['level_0']
    results[name]['bipartite_edgelist'] = (results[name]['bipartite_edgelist'].join(results[name]['bipartite_edgelist']
                                        [['reaction','source','target']].applymap(lambda x: results[name]['arrival_step'][x]).add_suffix('_step')))
    results[name]['impactful_edgelist'] =  (results[name]['bipartite_edgelist'].drop(results[name]['bipartite_edgelist']
                                                    .query('reaction==source & reaction_step>target_step').index).query('directed'))


# summarize results
def result_to_df(key: str, to_apply=lambda x: x):
    return pd.concat({i: results[i][key].apply(to_apply) for i in results.keys()}, axis=1)
summaries = {'compound count': result_to_df(key='cps', to_apply=lambda x: x.sum()).fillna(method='ffill'),
             'reaction count': result_to_df(key='rns', to_apply=lambda x: x.sum()).fillna(method='ffill'),
             'chiral fraction': result_to_df(key='chiral_fraction').fillna(method='ffill')}

# plot results
plot_expansion_summary(summaries, filename='kegg_R_seed_set_comparison.pdf')
plt.show()
plt.close()
# write output files
if write_outputs:
    write_edgelist_csvs(results=results)


In [ ]:
# Identify which compounds and reactions are important for the addition of chiral compounds in the network,
# as of the expansion step at which the network becomes majority-chiral. Unused seeds are ignored. Only
# forward-directed edges are considered - i.e. from a compound to a reaction to a *newer* compound.

transition_steps = summaries['chiral fraction'].where(summaries['chiral fraction'] >= 0.5).idxmin()
for name, transition_step in transition_steps.items():
    transition_edgelist = results[name]['impactful_edgelist'].query('step<=@transition_step & directed')
    G = nx.from_pandas_edgelist(transition_edgelist, source='source', target='target', edge_attr=True, create_using=nx.DiGraph())
    compound_nodes = [n for n in G.nodes if n.startswith('C')]
    seed_nodes_in_transition_network = [n for n in compound_nodes if n in results[name]['seeds']]
    nx.set_node_attributes(G, chiral_labels.loc[compound_nodes].T.to_dict())
    seed2chiral_bc = pd.Series(nx.betweenness_centrality_subset(G, sources=seed_nodes_in_transition_network, 
                                                      targets=list(chiral_labels.loc[compound_nodes].query('is_chiral').index))).sort_values(ascending=False)
    nx.set_node_attributes(G, seed2chiral_bc, 'seed2chiral_bc')
    results[name]['transition_network'] = G
    results[name]['transition_seed2chiral_bc_compounds'] = seed2chiral_bc.filter(like='C').where(lambda x: x>0).dropna(how='all')
    results[name]['transition_seed2chiral_bc_reactions'] = seed2chiral_bc.filter(like='R').where(lambda x: x>0).dropna(how='all')

summaries['transition_seed2chiral_bc_compounds'] = result_to_df(key='transition_seed2chiral_bc_compounds', to_apply=lambda x: x)
summaries['transition_seed2chiral_bc_reactions'] = result_to_df(key='transition_seed2chiral_bc_reactions', to_apply=lambda x: x) 

betweenness_chirality_corr = dict()
for name, centrality_values in summaries['transition_seed2chiral_bc_compounds'].items():
    betweenness_chirality_corr[name] = stats.pointbiserialr(centrality_values.dropna().values, chiral_labels['is_chiral'].loc[centrality_values.dropna().index])
betweenness_chirality_corr
